In [ ]:

!pip install -U ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.9 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Wed Sep  2 18:12:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

import os

for root, dirs, files in os.walk("/content/drive/MyDrive/GLOVES"):
    for file in files:
        if file == "data.yaml":
            print(os.path.join(root, file))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import zipfile
import os

zip_file = "/content/drive/MyDrive/glove.zip"
extract_path = "/content/glove_dataset"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [11]:

from ultralytics import YOLO
import yaml

DATA_YAML = "/content/glove_dataset/data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

print("Dataset:")
print(data)

print("\nClasses:")
print(data.get("names"))

Dataset:
{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'Gloves', 1: 'NO-Gloves'}}

Classes:
{0: 'Gloves', 1: 'NO-Gloves'}


In [12]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO("yolo26n.pt")

results = model.train(
    data=DATA_YAML,

    # Main training configuration
    epochs=50,
    imgsz=640,
    batch=16,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,

    # Warmup
    warmup_epochs=3,

    # Augmentation
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,

    # Validation
    val=True,

    # Save best model
    save=True,
    save_period=10,

    # Early stopping
    patience=50,

    # Experiment name
    project="safety_gloves",
    name="yolo26n_300epochs",

    # Use GPU
    device=0,

    # Workers
    workers=2,

    # Cache
    cache=False,

    verbose=True
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/glove_dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=yolo26n_300epochs, nbs=64, nms=

In [27]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/safety_gloves/yolo26n_300epochs/weights/best.pt")

results = model.predict(
    source="/content/glove_dataset/train/images/PP02img1131_jpg.rf.24d7f14da08147362268f696c2d51b54.jpg",
    conf=0.5,
    save=True
)


image 1/1 /content/glove_dataset/train/images/PP02img1131_jpg.rf.24d7f14da08147362268f696c2d51b54.jpg: 640x640 2 Glovess, 8.9ms
Speed: 1.1ms preprocess, 8.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict-15


In [ ]:
#hyper parameter tuning
model = YOLO("yolo26n.pt")
# Hyperparameter search space
search_space = {
    "lr0": (1e-5, 1e-2),
    "lrf": (0.01, 1.0),
    "momentum": (0.7, 0.98),
    "weight_decay": (0.0, 0.001),
    "warmup_epochs": (0.0, 5.0),
    "degrees": (0.0, 15.0),
    "translate": (0.0, 0.2),
    "scale": (0.3, 0.8),
    "shear": (0.0, 5.0),
    "perspective": (0.0, 0.001),
    "fliplr": (0.0, 0.5),
    "mosaic": (0.5, 1.0),
    "mixup": (0.0, 0.2),
}

results = model.tune(
    data=DATA_YAML,
    epochs=40,
    iterations=25,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    space=search_space,
    plots=True,
    save=True,
    val=True,
    name="glove_tuning"
)